## 異なるモデルの比較

このノートブックでは、これまで利用してきたtokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3と並行して別のモデルであるibm-granite/granite-7b-instructを使用し、その挙動を観察します。
ibm-granite/granite-7b-instructは英語を中心に学習されたモデルですが、日本語もある程度は処理することが可能です。両者のアウトプットの違いを確認してみましょう。

### 必要なライブラリとインポート

Labの指示に従って適切なワークベンチイメージを選択して起動した場合、必要なすべてのライブラリがすでにインストールされているはずです。もしインストールされていない場合は、次のセルの最初の行のコメントを外して正しいパッケージをすべてインストールしてください。その後、必要なライブラリをインポートします。

In [ ]:
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt # 正しいワークベンチイメージを選択していない場合のみコメントを外してください

import json

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchainパイプライン

Langchainを使用して、パイプラインを定義します。

In [ ]:
# LLM推論APIのURL
inference_server_url = "http://llama-3-1-swallow-8b-instruct-v0-3-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLMの定義
llm = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base= f"{inference_server_url}/v1",
    model_name="llama-3-1-swallow-8b-instruct-v0-3",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [ ]:
# Granite LLM推論サーバーURL
inference_server_url_granite = "http://granite-7b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLMの定義
llm_granite = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base= f"{inference_server_url_granite}/v1",
    model_name="granite-7b-instruct",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

それぞれのモデルに合わせた**テンプレート**を作成します。

In [ ]:
template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|>


あなたは、親切で、礼儀正しく、正直なアシスタントです。
常に気配りと尊重をもって接し、真摯にサポートします。できる限り有用な返答を提供しますが、安全を確保します。
有害で、倫理に反する、偏見のある、または否定的な内容は避けます。返答が公正でポジティブなものであることを確認します。<|eot_id|><|start_header_id|>user<|end_header_id|>

与えられた文章の内容をもとに、与えられた質問に答えてください。

### 文章:
{text}

### 質問:
{query}

### 回答:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

prompt = PromptTemplate(input_variables=["text", "query"], template=template)

In [ ]:
template="""<|system|>
あなたは、親切で、礼儀正しく、正直なアシスタントです。
常に気配りと尊重をもって接し、真摯にサポートします。できる限り有用な返答を提供しますが、安全を確保します。
有害で、倫理に反する、偏見のある、または否定的な内容は避けます。返答が公正でポジティブなものであることを確認します。

<|user|>
与えられた文章の内容をもとに、与えられた質問に答えてください。

### 文章:
{text}

### 質問:
{query}

### 回答:
<|assistant|>
"""
prompt_granite = PromptTemplate(input_variables=["text", "query"], template=template)

2つの**会話**オブジェクトを作成し、それぞれのモデルにクエリを投げる準備が行います。

In [ ]:
conversation = prompt | llm
conversation_granite = prompt_granite | llm_granite

モデルにクエリを投げる準備が整いました！

この例では、1つの請求文章を対象にクエリを実行しどのような結果が得られるかを見てみます。もちろん、他の請求文章で試してみても大丈夫です。

In [ ]:
filename = 'claims/claim1.json'

# Opening JSON file
claims = {}
with open(filename, 'r') as file:
    data = json.load(file)
claims[filename] = data

# Analyze the claim
print(f"***************************")
print(f"* 請求: {filename}")
print(f"***************************")
print("元の文章:")
print("-----------------")
print(f"件名: {claims[filename]['subject']}\n内容:\n{claims[filename]['content']}\n\n")
print('tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3による分析:')
print("--------")
text_input = f"件名: {claims[filename]['subject']}\n内容:\n{claims[filename]['content']}"
sentiment_query = "この請求の文章から読み取れる感情はどのようなものですか？「肯定的」、「否定的」、「どちらでもない」から1つだけ選んで答え、その理由もあわせて説明してください。"
location_query = "この請求に関連する出来事はどこで起こりましたか？出来事の発生した場所について、市区町村や通りの名前などを含めて答えて下さい。"
time_query = "この請求に関連する出来事はいつ起こりましたか？日付と、時刻あるいは時間帯を一言で答えて下さい。"
print(f"- 送信者の感情: ")
conversation.invoke(input={"text": text_input, "query": sentiment_query})
print("\n- 発生場所: ")
conversation.invoke(input={"text": text_input, "query": location_query})
print("\n- 発生日時: ")
conversation.invoke(input={"text": text_input, "query": time_query})
print("\n\n                          ----====----\n")
print('ibm-granite/granite-7b-instructによる分析:')
print("--------")
print(f"- 送信者の感情: ")
conversation_granite.invoke(input={"text": text_input, "query": sentiment_query})
print("\n- 発生場所: ")
conversation_granite.invoke(input={"text": text_input, "query": location_query})
print("\n- 発生日時: ")
conversation_granite.invoke(input={"text": text_input, "query": time_query})
print("\n\n                          ----====----\n")

両者の出力はどちらも正しいですが、日本語に特化したチューニングが行われたtokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.の方がより自然な回答となっています。

LLMを扱う際の技術は、求めるパフォーマンスと精度の間、ならびにそれに伴うリソースやコストとのバランスを見つけることです。

そのため、データが変化したり、モデルが進化したりする際に、常に期待通りの挙動が得られるようにするための信頼性チェックを行うことが重要となります。